<a href="https://colab.research.google.com/github/zhangyingchengqi/Modern-Computer-Vision-with-PyTorch/blob/master/Chapter02/Implementing_custom_loss_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第二章 第七节 自定义损失函数

回归 → MSE（数据干净）、MAE（异常值多）、Huber（折中）。

二分类 → BCEWithLogitsLoss（数值稳定）。

多分类 → CrossEntropyLoss（默认选择）。

类别不平衡 → Focal Loss。

度量学习 → Triplet Loss / Contrastive Loss。

生成模型 → KL Loss、Wasserstein Loss。




In [ ]:
# ===================== 数据准备 =====================

# 原始输入数据（特征），4 行代表 4 个样本，每行有 2 个特征
x = [[1, 2], [3, 4], [5, 6], [7, 8]]
# 原始输出数据（标签），4 行代表 4 个样本，每行 1 个输出值
y = [[3], [7], [11], [15]]
import torch
# 将 Python 列表转换为 PyTorch 张量，并转为 float 类型（浮点数计算）
X = torch.tensor(x).float()
Y = torch.tensor(y).float()
import torch.nn as nn
# 如果当前机器有 GPU，则使用 GPU（cuda），否则使用 CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# 将数据移动到对应的计算设备（CPU 或 GPU）
X = X.to(device)
Y = Y.to(device)
from torch.utils.data import Dataset, DataLoader
# ===================== 自定义数据集类 =====================
class MyDataset(Dataset):
    def __init__(self, x, y):
        # 初始化时保存输入特征和标签，转成 float 类型
        self.x = torch.tensor(x).float()
        self.y = torch.tensor(y).float()
    def __len__(self):
        # 返回数据集中样本的总数量
        return len(self.x)
    def __getitem__(self, ix):
        # 根据索引 ix 返回对应的 (特征, 标签) 元组
        return self.x[ix], self.y[ix]
# 创建数据集对象
ds = MyDataset(X, Y)
# 使用 DataLoader 按 batch_size=2 加载数据，并打乱顺序（shuffle=True）
dl = DataLoader(ds, batch_size=2, shuffle=True)
# ===================== 定义神经网络模型 =====================
class MyNeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 输入层 -> 隐藏层（输入特征数为 2，隐藏层单元数为 8）
        self.input_to_hidden_layer = nn.Linear(2, 8)
        # 隐藏层激活函数，使用 ReLU（Rectified Linear Unit）
        self.hidden_layer_activation = nn.ReLU()
        # 隐藏层 -> 输出层（隐藏层单元数为 8，输出特征数为 1）
        self.hidden_to_output_layer = nn.Linear(8, 1)

    def forward(self, x):
        # 前向传播：依次通过输入层、激活函数、输出层
        x = self.input_to_hidden_layer(x)   # 线性变换
        x = self.hidden_layer_activation(x) # 非线性激活
        x = self.hidden_to_output_layer(x)  # 最终输出
        return x
# 创建模型对象，并将其移动到指定计算设备
mynet = MyNeuralNet().to(device)

/tmp/ipython-input-2056695478.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.x = torch.tensor(x).float()
/tmp/ipython-input-2056695478.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.y = torch.tensor(y).float()


In [ ]:
# 自定义一个 MSE 损失函数
def my_mean_squared_error(_y, y):
    loss = (_y-y)**2  #取差的平方
    loss = loss.mean() #均值
    return loss

In [ ]:
#使用 pytorch提供的MSE损失函数计算
loss_func = nn.MSELoss()
loss_value = loss_func(mynet(X),Y)
print(loss_value)

tensor(125.1687, grad_fn=<MseLossBackward0>)


In [ ]:
#使用自定义的损失函数   结果是一样的，说明 自定义的函数是正确的
my_mean_squared_error(mynet(X),Y)

tensor(125.1687, grad_fn=<MeanBackward0>)

## 下一节 第二章第八节 获取中间层的参数值 Fetching_values_of_intermediate_layers